In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Procesa Bronze
# MAGIC
# MAGIC Lee archivos desde el volume `raw` usando Auto Loader y escribe en:
# MAGIC
# MAGIC - `bronze_reportes`
# MAGIC - `control_archivos`
# MAGIC
# MAGIC Reglas:
# MAGIC - Solo procesa archivos con nombre `report_<numero>.txt`.
# MAGIC - Usa Unity Catalog.
# MAGIC - Usa `_metadata.file_path` en lugar de `input_file_name()`.
# MAGIC - No usa `cache()` ni `persist()` porque no está soportado en serverless dentro de `foreachBatch`.

In [0]:
# COMMAND ----------

import re
from pyspark.sql import functions as F

dbutils.widgets.removeAll()

dbutils.widgets.text("catalog_name", "smart_data_dev", "Catalog name")
dbutils.widgets.text("schema_name", "storage", "Schema name")
dbutils.widgets.text("storage_account", "storagetoxdev", "Storage account")
dbutils.widgets.text("id_proceso", "02ae3d8f-05ac-4af0-86e8-05762dc2db18", "Process id")

catalog_name = dbutils.widgets.get("catalog_name").strip()
schema_name = dbutils.widgets.get("schema_name").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
id_proceso = dbutils.widgets.get("id_proceso").strip()

if not all([catalog_name, schema_name, storage_account, id_proceso]):
    raise ValueError(
        "catalog_name, schema_name, storage_account e id_proceso son obligatorios."
    )

tabla_control_name = f"{catalog_name}.{schema_name}.control_archivos"
tabla_bronze_name = f"{catalog_name}.{schema_name}.bronze_reportes"

tabla_control_sql = f"`{catalog_name}`.`{schema_name}`.`control_archivos`"
tabla_bronze_sql = f"`{catalog_name}`.`{schema_name}`.`bronze_reportes`"

print(f"catalog_name: {catalog_name}")
print(f"schema_name: {schema_name}")
print(f"storage_account: {storage_account}")
print(f"id_proceso: {id_proceso}")
print(f"tabla_control_name: {tabla_control_name}")
print(f"tabla_bronze_name: {tabla_bronze_name}")

In [0]:
# COMMAND ----------

raw_base_path = f"/Volumes/{catalog_name}/{schema_name}/raw"

checkpoint_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "metadata/smart_data_ingestion/checkpoints/raw_to_bronze"
)

schema_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "metadata/smart_data_ingestion/schemas/raw_to_bronze"
)

print(f"raw_base_path: {raw_base_path}")
print(f"checkpoint_path: {checkpoint_path}")
print(f"schema_path: {schema_path}")

In [0]:
# COMMAND ----------

# Validación rápida de archivos visibles en raw.
# Esta celda es útil para debug manual y no modifica datos.

archivos_raw = dbutils.fs.ls(raw_base_path)

patron_archivo_python = re.compile(r"^report_\d+\.txt$")

for archivo in archivos_raw:
    nombre_archivo = archivo.name.rstrip("/")
    cumple_patron = bool(patron_archivo_python.match(nombre_archivo))
    print(f"{nombre_archivo} -> {cumple_patron}")

In [0]:
# COMMAND ----------

# Auto Loader usa glob como prefiltro. No es regex.
# Luego aplicamos regex exacto sobre nombre_archivo.

patron_archivo_regex = r"^report_\d+\.txt$"
patron_archivo_glob = "report_*.txt"

df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.useStrictGlobber", "true")
    .option("pathGlobFilter", patron_archivo_glob)
    .option("header", "true")
    .option("delimiter", ",")
    .option("mode", "PERMISSIVE")
    .option("rescuedDataColumn", "_rescued_data")
    .load(raw_base_path)
)

In [0]:
# COMMAND ----------

# Normalizamos nombres de columnas del archivo fuente hacia el schema bronze esperado.
# Los archivos reales usan encabezados como: email,jk,Badmail,Baja,Fecha envio,...

df_bronze = (
    df_raw
    .withColumnRenamed("jk", "jyv")
    .withColumnRenamed("Badmail", "badmail")
    .withColumnRenamed("Baja", "baja")
    .withColumnRenamed("Fecha envio", "fecha_envio")
    .withColumnRenamed("Fecha open", "fecha_open")
    .withColumnRenamed("Opens", "opens")
    .withColumnRenamed("Opens virales", "opens_virales")
    .withColumnRenamed("Fecha click", "fecha_click")
    .withColumnRenamed("Clicks", "clicks")
    .withColumnRenamed("Clicks virales", "clicks_virales")
    .withColumnRenamed("Links", "links")
    .withColumnRenamed("IPs", "ips")
    .withColumnRenamed("Navegadores", "navegadores")
    .withColumnRenamed("Plataformas", "plataformas")
    .withColumn("archivo_origen", F.col("_metadata.file_path"))
    .withColumn(
        "nombre_archivo",
        F.regexp_extract(F.col("archivo_origen"), r"([^/]+)$", 1)
    )
    .filter(F.col("nombre_archivo").rlike(patron_archivo_regex))
    .withColumn("id_proceso", F.lit(id_proceso))
    .withColumn("fecha_ejecucion", F.current_timestamp())
)

if "_corrupt_record" not in df_bronze.columns:
    df_bronze = df_bronze.withColumn("_corrupt_record", F.lit(None).cast("string"))

if "_rescued_data" not in df_bronze.columns:
    df_bronze = df_bronze.withColumn("_rescued_data", F.lit(None).cast("string"))

columnas_bronze = [
    "email",
    "jyv",
    "badmail",
    "baja",
    "fecha_envio",
    "fecha_open",
    "opens",
    "opens_virales",
    "fecha_click",
    "clicks",
    "clicks_virales",
    "links",
    "ips",
    "navegadores",
    "plataformas",
    "_corrupt_record",
    "_rescued_data",
    "archivo_origen",
    "nombre_archivo",
    "id_proceso",
    "fecha_ejecucion",
]

df_bronze = df_bronze.select(*columnas_bronze)

In [0]:
# COMMAND ----------

def escribir_bronze_y_control(batch_df, batch_id):
    total_registros = batch_df.count()

    print(f"batch_id: {batch_id}")
    print(f"registros recibidos: {total_registros}")

    if total_registros == 0:
        print("Batch vacío. No se escribe nada.")
        return

    (
        batch_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(tabla_bronze_name)
    )

    archivos_df = (
        batch_df
        .groupBy("nombre_archivo", "archivo_origen")
        .agg(F.count(F.lit(1)).alias("registros_bronze"))
        .withColumn("registros_invalidos_bronze", F.lit(None).cast("bigint"))
        .withColumn("registros_silver", F.lit(None).cast("bigint"))
        .withColumn("registros_error", F.lit(None).cast("bigint"))
        .withColumn("fecha_ejecucion", F.current_timestamp())
        .withColumn("id_proceso", F.lit(id_proceso))
        .withColumn("detalle", F.lit("Archivo cargado en bronze"))
        .select(
            "nombre_archivo",
            "archivo_origen",
            "registros_bronze",
            "registros_invalidos_bronze",
            "registros_silver",
            "registros_error",
            "fecha_ejecucion",
            "id_proceso",
            "detalle"
        )
    )

    (
        archivos_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(tabla_control_name)
    )

    print("Escritura bronze y control completada.")

In [0]:
# COMMAND ----------

query = (
    df_bronze
    .writeStream
    .foreachBatch(escribir_bronze_y_control)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# COMMAND ----------

print("Validación de registros cargados para este id_proceso")

display(
    spark.table(tabla_bronze_name)
    .filter(F.col("id_proceso") == id_proceso)
)

display(
    spark.table(tabla_control_name)
    .filter(F.col("id_proceso") == id_proceso)
)